Récupération des données

On obtient la cle api de météo france à l'aide de getpass (interactif avec celui qui lance la commande).
Le .strip() est fait pour supprimer les espace et les sauts de lignes du mot (ça évite des erreurs lors du copié collé)

!!!il faudrait donner le lien du site ou obtenir une clé!!!

Choisir les stations avec liste-stations/quotidienne.
Commander les données d’une station avec commande-station/quotidienne, qui renvoie un numéro de commande.
Récupérer le fichier avec commande/fichier en donnant ce numéro.

In [ ]:
import requests 
import time
from getpass import getpass
import pandas as pd
import io

url_stations = 'https://public-api.meteofrance.fr/public/DPClim/v1/liste-stations/quotidienne?id-departement=95&parametre=vent'

CLE_API = getpass("Clé API Météo-France : ").strip()

'''
Récupération de l'id des stations d'intérêt
'''
id_stations = []

r_stations = requests.get(url_stations, headers={"apikey": CLE_API})
liste_stations = r_stations.json()
id_stations += [liste_stations[i]['id'] for i in range(len(liste_stations))]
print(id_stations)

['95078001', '95088001', '95306001', '95527001', '95580001', '95678001', '95690001']


In [29]:
'''
Récupération du code commande associé à chaque station
'''

codes_commande = []
for station in id_stations:

    url_code_commande = f'https://public-api.meteofrance.fr/public/DPClim/v1/commande-station/quotidienne?id-station={station}&date-deb-periode=2025-01-01T00%3A00%3A00Z&date-fin-periode=2025-12-31T23%3A59%3A59Z'
    r_code_commande = requests.get(url_code_commande, headers = {"apikey": CLE_API})
    codes_commande.append(r_code_commande.json()['elaboreProduitAvecDemandeResponse']['return'])
    print(r_code_commande)
print(codes_commande)

<Response [202]>
<Response [202]>
<Response [202]>
<Response [202]>
<Response [202]>
<Response [202]>
<Response [202]>
['2026042723378', '2026042723381', '2026042723382', '2026042723384', '2026042723385', '2026042723387', '2026042723388']


Certaines stations sont actuellement fermées donc pour certaines périodes il se peut qu'il n'y ait pas de données. Bien que l'API commande nous donne bien un code pour télécharger le fichier, lorsque l'on essaye l'API téléchargement avec, ça nous renvoie un statut 404 (pas de ressources trouvées), ce qui fait planter le code. Pour contourner cela, on vérifie à chaque fois que le statut est bien 201 (requête réussite).

In [ ]:
'''
Récupération des fichiers des données des commandes
'''

time.sleep(3)  #on attend 3 secondes que les fichiers soient créés
liste_df = []
variables_vent = ['POSTE', 'DATE', 'FFM', 'FF2M', 'FXY', 'DXY', 'FXI','DXI', 'FXI2', 'DXI2', 'FXI3S', 'DXI3S']
for code in codes_commande:

    url_telechargement_donnees = f'https://public-api.meteofrance.fr/public/DPClim/v1/commande/fichier?id-cmde={code}'
    r_telechargement_donnees = requests.get(url_telechargement_donnees, headers = {'apikey': CLE_API})
    print(r_telechargement_donnees)
    if r_telechargement_donnees.status_code == 201:
        df_donnees = pd.read_csv(io.StringIO(r_telechargement_donnees.text), sep=";", decimal=",",usecols=variables_vent, dtype={"POSTE": str}, parse_dates=["DATE"], date_format="%Y%m%d")  # garde les zéros initiaux de l'identifiant (POSTE)
        liste_df.append(df_donnees)

df_donnees_entier = pd.concat(liste_df, ignore_index=True)
df_donnees_entier

<Response [201]>
<Response [201]>
<Response [404]>
<Response [201]>
<Response [201]>
<Response [404]>
<Response [201]>


,POSTE,DATE,FFM,FXI,DXI,FXY,DXY,FF2M,FXI2,DXI2,FXI3S,DXI3S
0,95078001,2025-01-01,8.2,18.3,220.0,11.6,220.0,NaN,NaN,NaN,17.2,210.0
1,95078001,2025-01-02,2.7,7.5,360.0,6.1,280.0,NaN,NaN,NaN,7.2,360.0
2,95078001,2025-01-03,1.4,5.5,350.0,3.7,330.0,NaN,NaN,NaN,5.2,350.0
3,95078001,2025-01-04,3.4,8.6,120.0,5.7,120.0,NaN,NaN,NaN,8.3,120.0
4,95078001,2025-01-05,7.5,18.5,220.0,10.6,210.0,NaN,NaN,NaN,16.2,220.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1820,95690001,2025-12-27,3.7,9.4,80.0,5.9,80.0,NaN,NaN,NaN,8.6,80.0
1821,95690001,2025-12-28,3.7,10.5,70.0,6.3,50.0,NaN,NaN,NaN,9.1,40.0
1822,95690001,2025-12-29,3.3,10.0,60.0,6.3,40.0,NaN,NaN,NaN,9.6,60.0
1823,95690001,2025-12-30,3.4,8.4,40.0,5.1,40.0,NaN,NaN,NaN,7.6,40.0
